In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, date_format, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType
import os

kafka_bootstrap_servers = "kafka:9092" #os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:9093")
kafka_topic = os.environ.get("KAFKA_TOPIC", "instagram-post")

# Configurações do S3/MinIO
s3_bucket_name = "warehouse" # Nome do seu bucket no MinIO (criado pelo `mc` do docker-compose)
output_path = f"s3a://{s3_bucket_name}/bronze/instagram/" # Caminho de saída no MinIO


In [2]:
from pyspark.sql import SparkSession
# .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
# .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
# .config("spark.sql.catalog.spark_catalog.type", "hive") \
# .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
# .config("spark.sql.catalog.local.type", "hadoop") \
    # .config("spark.hadoop.fs.s3a.experimental.input.fadvise", "sequential") \
spark = SparkSession.builder \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5,org.apache.hadoop:hadoop-client:3.3.6,com.amazonaws:aws-java-sdk-bundle:1.12.593") \
    .config("spark.sql.catalog.local.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.experimental.input.fadvise", "sequential") \
.getOrCreate()

In [3]:
spark._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://minio:9000")
spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", "admin")
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "password")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

In [4]:
spark.range(10).write.mode("overwrite").parquet("s3a://warehouse/teste_s3a")


Py4JJavaError: An error occurred while calling o52.parquet.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2688)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWritingFileFormat(DataSource.scala:454)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWriting(DataSource.scala:530)
	at org.apache.spark.sql.DataFrameWriter.saveToV1Source(DataFr

In [5]:
sc._jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())

NameError: name 'sc' is not defined

In [ ]:
spark._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark.range(5).write.mode("overwrite").parquet("s3a://warehouse/teste_s3a")

In [ ]:
# files = spark._jvm.org.apache.hadoop.fs.FileSystem \
#     .get(spark._jsc.hadoopConfiguration()) \
#     .listStatus(spark._jvm.org.apache.hadoop.fs.Path("s3a://warehouse"))

In [ ]:
from pyspark.sql import SparkSession
# spark = SparkSession.builder \
#     .appName("KafkaTest") \
#     .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5") \
#     .getOrCreate()
# Leitura do tópico Kafka
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()




In [ ]:
comment_schema = StructType([
    StructField("user", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", StringType(), True)
])
# Schema do JSON enviado
instagram_schema = StructType([
    StructField("id", StringType(), True),
    StructField("user_handle", StringType(), True),
    StructField("caption", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("posted_at", StringType(), True),  # ou TimestampType se já estiver parseado
    StructField("likes", IntegerType(), True),
    StructField("hashtags", ArrayType(StringType()), True),
    StructField("comments", ArrayType(comment_schema), True)
])

In [ ]:

# Extração e parsing do JSON
df_parsed = df_raw.selectExpr("CAST(value AS STRING) as json") \
    .withColumn("data", from_json(col("json"), instagram_schema)) \
    .select("data.*")
df_parsed = df_parsed.withColumn("event_time", current_timestamp())\
    .withColumn("ingestion_date", date_format(col("event_time"), "yyyy-MM-dd"))

# Escrever os dados brutos na camada Bronze do Data Lake
query = (df_parsed.coalesce(1).writeStream\
    .format("parquet")\
    .partitionBy("ingestion_date")\
    .option("path", output_path)\
    .option("checkpointLocation", f"s3a://{s3_bucket_name}/_spark_checkpoint/instagram/") \
    .trigger(processingTime="10 minutes")\
    .outputMode("append")\
    .start())


In [ ]:

query.awaitTermination()